# Buổi H2: Hàm dự đoán an toàn, kiểm tra đầu vào, trả lời kèm lý do

**Hiện trường.** Mô hình đã nạp được (H1). Nhân viên nhập thông tin khách bằng tay: có người bỏ trống ô, có người gõ "-5" tháng gắn bó, có người ghi thành phố mới chưa từng thấy. Nếu mô hình cứ nuốt hết và trả ra một con số, không ai biết nó đang đoán trên dữ liệu vô lý. Chị Hạnh cần một **hàm dự đoán** biết từ chối khi đầu vào sai, biết nói "tôi không chắc", và giải thích vì sao.

Buổi này dạy viết hàm dự đoán gồm: **lược đồ đầu vào** (cột nào, kiểu gì, khoảng nào), **kiểm tra** trước khi dự đoán, **trả lời có cấu trúc** (xác suất, nhãn, lý do, cảnh báo), **ngưỡng quyết định theo chi phí** (C2, E7), và **nhật ký** để theo dõi ở H3.

**Ẩn dụ:** quầy lễ tân bệnh viện. Trước khi bác sĩ khám, lễ tân kiểm giấy tờ: thiếu thì bảo bổ sung, vô lý thì hỏi lại. Bác sĩ (mô hình) chỉ nhận ca đã qua cửa kiểm.

**Mindset của người làm dữ liệu:** mô hình chỉ đáng tin trong vùng dữ liệu nó từng thấy. Hàm dự đoán tốt biết **ranh giới đó** và nói ra.

**Lưu ý:** dữ liệu khách hàng dựng có chủ đích; dùng để học.

## Bản đồ các chặng

| Chặng | Câu hỏi của chặng |
|---|---|
| Chặng 0 | Thấy ngay: mô hình nuốt cả dữ liệu vô lý |
| Chặng 1 | Lược đồ và hàm kiểm tra |
| Chặng 2 | Trả lời có cấu trúc: xác suất, nhãn, lý do |
| Chặng 3 | Một dòng và nhiều dòng phải cho cùng kết quả |
| Chặng 4 | Ngưỡng quyết định theo chi phí |
| Chặng 5 | Nhật ký và xử lý khi đầu vào sai |

## Chặng 0: Thấy ngay, mô hình nuốt dữ liệu vô lý

**Câu hỏi:** đưa vào những dòng vô lý, mô hình làm gì?

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
BLUE, GRAY = "#1F4E79", "#8C8C8C"
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

**Giải thích từng dòng**

- `import numpy as np`: mượn thư viện numpy (máy tính bỏ túi cho cả dãy số), đặt tên ngắn là `np` để gõ cho nhanh.
- `import pandas as pd`: mượn pandas, thư viện làm việc với bảng số liệu giống Excel, tên ngắn là `pd`.
- `import matplotlib.pyplot as plt`: mượn công cụ vẽ biểu đồ, tên ngắn là `plt`.
- `BLUE, GRAY = "#1F4E79", "#8C8C8C"`: gán hai màu dùng suốt bài (xanh đậm và xám). Một dòng gán cùng lúc hai biến, mỗi biến một mã màu.
- `plt.rcParams["figure.figsize"] = (6, 4)`: đặt cỡ hình mặc định: rộng 6, cao 4 (đơn vị inch).
- `plt.rcParams["axes.spines.top"] = False`: bỏ đường viền phía trên khung hình cho gọn mắt.
- `plt.rcParams["axes.spines.right"] = False`: bỏ luôn đường viền bên phải.

In [2]:
def make_customers(n, seed, month=0):
    """Synthetic customer table (built on purpose for teaching). month > 0 makes later data drift."""
    r = np.random.RandomState(seed)
    tenure = r.gamma(3.0, 6.0, n).round(0).clip(1, 72)
    plan = r.choice(["basic", "plus", "premium"], n, p=[0.55 - 0.03 * month, 0.30, 0.15 + 0.03 * month])
    city = r.choice(["HCM", "HN", "DN", "CT"], n, p=[0.4, 0.3, 0.15, 0.15])
    spend = (np.where(plan == "basic", 0.8, np.where(plan == "plus", 1.6, 3.0)) * r.lognormal(0, 0.35, n) * (1 + 0.04 * month)).round(2)
    visits = r.poisson(np.clip(2 + 0.12 * tenure ** 0.7 + 0.5 * spend, 0.5, None)).astype(int)
    complaints = r.poisson(0.35 + 0.002 * month, n).astype(int)
    z = -0.2 + 0.035 * tenure + 0.32 * visits - 0.55 * complaints + 0.25 * spend - 1.6
    if month:
        z = z - 0.12 * month * (plan == "premium")
    returned = (r.rand(n) < 1 / (1 + np.exp(-z))).astype(int)
    df = pd.DataFrame({"tenure_months": tenure, "monthly_spend": spend, "visits_last_30d": visits, "complaints": complaints, "plan": plan, "city": city, "returned": returned})
    miss = r.rand(n) < 0.04
    df.loc[miss, "monthly_spend"] = np.nan
    return df

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
df = make_customers(4000, seed=0)
X, y = df.drop(columns="returned"), df["returned"]
X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_rest, y_rest, test_size=0.5, random_state=0, stratify=y_rest)
num_cols = ["tenure_months", "monthly_spend", "visits_last_30d", "complaints"]
cat_cols = ["plan", "city"]
prep = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
model = Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=1000))]).fit(X_train, y_train)
normal = X_test.iloc[[0]].copy()
cases = {
    "normal row": normal,
    "tenure = -5 months": normal.assign(tenure_months=-5),
    "tenure = 5000 months": normal.assign(tenure_months=5000),
    "unknown city 'PQ'": normal.assign(city="PQ"),
    "missing spend": normal.assign(monthly_spend=np.nan),
    "column 'city' missing": normal.drop(columns="city"),
}
rows = {}
for name, row in cases.items():
    try:
        rows[name] = round(float(model.predict_proba(row)[0, 1]), 3)
    except Exception as e:
        rows[name] = "ERROR: " + type(e).__name__
print(pd.Series(rows).to_string())

normal row                            0.87
tenure = -5 months                    0.79
tenure = 5000 months                   1.0
unknown city 'PQ'                     0.87
missing spend                        0.559
column 'city' missing    ERROR: ValueError


**Giải thích từng dòng**

- `make_customers(4000, seed=0)`: bảng khách hàng giả như H1; chia ba phần: học, kiểm (để chọn ngưỡng ở chặng 4) và thi.
- `OneHotEncoder(handle_unknown="ignore")`: gặp giá trị chữ lạ thì để toàn số 0 thay vì báo lỗi.
- `normal.assign(tenure_months=-5)`: tạo một dòng giống dòng bình thường nhưng đổi một giá trị thành vô lý.
- `try: ... except Exception`: thử dự đoán, nếu lỗi thì ghi lại tên lỗi thay vì dừng chương trình.

**Cách đọc.**
- **Con số:** dòng bình thường cho **0.87**; thời gian gắn bó âm cho **0.79**; 5000 tháng cho **1.0**; thành phố lạ cho **0.87**; thiếu chi tiêu cho **0.559**; thiếu cột thành phố cho **ERROR: ValueError**.
- **Bức tranh:** mô hình **vui vẻ nuốt** giá trị âm và giá trị 5000 tháng (hơn 400 năm) rồi trả ra một xác suất trông rất nghiêm túc. Chỉ có cột bị thiếu hẳn mới gây lỗi. Ô trống và thành phố lạ được xử lý có chủ đích bởi pipeline.
- **Câu hỏi tiếp:** ai chặn các giá trị vô lý? Không phải mô hình: phải là hàm bọc nó.

**Mang về chặng 1:** mô hình không biết dữ liệu nào vô lý; phải có người gác cửa.

## Chặng 1: Lược đồ và hàm kiểm tra

**Câu hỏi:** "đầu vào hợp lệ" nghĩa là gì, và viết nó ra thế nào?

In [3]:
SCHEMA = {
    "tenure_months":   {"kind": "number", "min": 0, "max": 120, "required": True},
    "monthly_spend":   {"kind": "number", "min": 0, "max": 50, "required": False},
    "visits_last_30d": {"kind": "number", "min": 0, "max": 60, "required": True},
    "complaints":      {"kind": "number", "min": 0, "max": 30, "required": True},
    "plan":            {"kind": "category", "allowed": ["basic", "plus", "premium"], "required": True},
    "city":            {"kind": "category", "allowed": ["HCM", "HN", "DN", "CT"], "required": True},
}
def validate_row(row):
    """Return (errors, warnings). Errors mean: refuse. Warnings mean: answer, but say it."""
    errors, warnings_ = [], []
    for col, rule in SCHEMA.items():
        if col not in row or row[col] is None or (isinstance(row[col], float) and np.isnan(row[col])):
            (errors if rule["required"] else warnings_).append(f"{col}: missing")
            continue
        v = row[col]
        if rule["kind"] == "number":
            if not isinstance(v, (int, float, np.integer, np.floating)):
                errors.append(f"{col}: not a number ({v!r})")
            elif not (rule["min"] <= v <= rule["max"]):
                errors.append(f"{col}: {v} outside [{rule['min']}, {rule['max']}]")
        elif v not in rule["allowed"]:
            warnings_.append(f"{col}: unseen value {v!r}")
    extra = [c for c in row if c not in SCHEMA]
    if extra:
        warnings_.append(f"ignored extra columns: {extra}")
    return errors, warnings_
checks = {name: validate_row(df_.iloc[0].to_dict()) for name, df_ in cases.items() if "missing" not in name or "spend" in name}
checks["column 'city' missing"] = validate_row(cases["column 'city' missing"].iloc[0].to_dict())
for name, (e, w) in checks.items():
    print(f"{name:24s} errors={e} warnings={w}")

normal row               errors=[] warnings=[]
tenure = -5 months       errors=['tenure_months: -5 outside [0, 120]'] warnings=[]
tenure = 5000 months     errors=['tenure_months: 5000 outside [0, 120]'] warnings=[]
unknown city 'PQ'        errors=[] warnings=["city: unseen value 'PQ'"]
missing spend            errors=[] warnings=['monthly_spend: missing']
column 'city' missing    errors=['city: missing'] warnings=[]


**Giải thích từng dòng**

- `SCHEMA = {...}`: **lược đồ**: với mỗi cột, kiểu, khoảng cho phép (hoặc danh sách giá trị cho phép), và có bắt buộc không. Khoảng lấy từ **hiểu biết nghiệp vụ** (không ai gắn bó 400 năm) chứ không chỉ từ dữ liệu học.
- `(errors if rule["required"] else warnings_).append(...)`: thiếu cột **bắt buộc** thì là lỗi (từ chối); thiếu cột **không bắt buộc** (pipeline điền được) thì chỉ cảnh báo.
- `rule["min"] <= v <= rule["max"]`: kiểm khoảng hợp lý.
- `v not in rule["allowed"]`: giá trị chữ lạ: **cảnh báo**, không từ chối, vì pipeline xử lý được nhưng dự đoán kém tin cậy hơn.

**Cách đọc.**
- **Con số:** dòng bình thường: ([], []); tháng gắn bó âm: (['tenure_months: -5 outside [0, 120]'], []); thành phố lạ: ([], ["city: unseen value 'PQ'"]); thiếu chi tiêu: ([], ['monthly_spend: missing']); thiếu cột thành phố: (['city: missing'], []).
- **Bức tranh:** hai mức phản ứng: **lỗi** (từ chối trả lời, vì kết quả vô nghĩa) và **cảnh báo** (trả lời nhưng nói rõ điều đáng ngờ). Phân biệt hai mức là quyết định nghiệp vụ, không phải kỹ thuật.
- **Quy tắc đọc:** lược đồ phải nằm **trong cùng thư mục với mô hình** (H1), được sửa cùng lúc khi huấn luyện lại.

**Mang về chặng 2:** viết lược đồ ra, kiểm tra trước khi dự đoán, phân biệt lỗi và cảnh báo.

## Chặng 2: Trả lời có cấu trúc

**Câu hỏi:** trả về cái gì thì người dùng hiểu và hành động được?

In [4]:
feature_names = model.named_steps["prep"].get_feature_names_out()
coef = model.named_steps["clf"].coef_[0]
THRESHOLD = 0.5
def predict_one(row, threshold=None):
    errors, warns = validate_row(row)
    if errors:
        return {"status": "refused", "errors": errors}
    threshold = THRESHOLD if threshold is None else threshold
    frame = pd.DataFrame([row])[list(SCHEMA)]
    proba = float(model.predict_proba(frame)[0, 1])
    transformed = model.named_steps["prep"].transform(frame)
    transformed = transformed.toarray() if hasattr(transformed, "toarray") else transformed
    contrib = pd.Series(transformed[0] * coef, index=feature_names)
    top = contrib.reindex(contrib.abs().sort_values(ascending=False).index).head(3)
    return {
        "status": "ok",
        "probability_returns": round(proba, 3),
        "decision": "likely returns" if proba >= threshold else "at risk of not returning",
        "threshold": threshold,
        "main_reasons": {k.replace("num__", "").replace("cat__", ""): round(float(v), 2) for k, v in top.items()},
        "warnings": warns,
    }
example = normal.iloc[0].to_dict()
risk = {**example, "tenure_months": 2, "visits_last_30d": 0, "complaints": 3}
print("normal row    ->", predict_one(example))
print("risky customer->", predict_one(risk))
print("bad input     ->", predict_one({**example, "tenure_months": -5}))

normal row    -> {'status': 'ok', 'probability_returns': 0.87, 'decision': 'likely returns', 'threshold': 0.5, 'main_reasons': {'monthly_spend': 1.58, 'visits_last_30d': 0.51, 'complaints': -0.29}, 'warnings': []}
risky customer-> {'status': 'ok', 'probability_returns': 0.248, 'decision': 'at risk of not returning', 'threshold': 0.5, 'main_reasons': {'monthly_spend': 1.58, 'visits_last_30d': -1.25, 'complaints': -1.2}, 'warnings': []}
bad input     -> {'status': 'refused', 'errors': ['tenure_months: -5 outside [0, 120]']}


**Giải thích từng dòng**

- `def predict_one(row, threshold=None)`: hàm dự đoán cho **một khách**, nhận một từ điển (như dữ liệu từ biểu mẫu).
- `if errors: return {"status": "refused", ...}`: đầu vào lỗi thì từ chối **có nói lý do**, không đoán bừa.
- `transformed[0] * coef`: với hồi quy logistic, mỗi cột góp vào kết quả một phần bằng **giá trị đã chuẩn hóa nhân hệ số**. Phần góp dương đẩy về "quay lại", âm đẩy về "không quay lại".
- `contrib.abs().sort_values(...).head(3)`: ba lý do có sức nặng lớn nhất, bất kể dương hay âm.
- `"warnings": warns`: trả luôn cảnh báo (cột thiếu không bắt buộc, giá trị lạ) để người dùng biết độ tin cậy.

**Cách đọc.**
- **Con số:** khách bình thường: xác suất quay lại **0.87**; khách rủi ro (gắn bó 2 tháng, 0 lần ghé, 3 khiếu nại): **0.248**, quyết định "at risk of not returning", lý do chính **['monthly_spend', 'visits_last_30d', 'complaints']**. Đầu vào sai nhận **refused**.
- **Bức tranh:** câu trả lời có bốn phần: **xác suất**, **quyết định**, **lý do**, **cảnh báo**. Với người dùng, "khách này có nguy cơ không quay lại vì ít ghé và nhiều khiếu nại" hữu ích hơn một con số trần.
- **Lưu ý:** cách tính lý do này đúng cho hồi quy logistic. Với mô hình khác (rừng, mạng) phải dùng cách khác (độ quan trọng hoán vị ở E6, hoặc SHAP).

**Mang về chặng 3:** trả lời có cấu trúc: xác suất, quyết định, lý do, cảnh báo.

## Chặng 3: Một dòng và nhiều dòng phải cho cùng kết quả

**Câu hỏi:** hàm xử lý từng khách một và hàm xử lý cả bảng có đồng nhất không? Cái nào nhanh hơn?

In [5]:
import time
sample = X_test.head(500)
t0 = time.time()
one_by_one = np.array([predict_one(r.to_dict())["probability_returns"] for _, r in sample.iterrows()])
t_one = time.time() - t0
t0 = time.time()
batch = model.predict_proba(sample)[:, 1].round(3)
t_batch = time.time() - t0
print("max difference one-by-one vs batch:", float(np.abs(one_by_one - batch).max()))
print("time for 500 rows -> one by one:", round(t_one, 2), "s | batch:", round(t_batch, 4), "s | ratio:", round(t_one / t_batch))

max difference one-by-one vs batch: 0.0
time for 500 rows -> one by one: 4.97 s | batch: 0.0062 s | ratio: 798


**Giải thích từng dòng**

- `for _, r in sample.iterrows()`: gọi `predict_one` lần lượt cho 500 khách, mỗi lần kiểm tra, dự đoán, tính lý do.
- `model.predict_proba(sample)`: dự đoán cả 500 dòng trong một lần gọi.
- `np.abs(one_by_one - batch).max()`: hai cách phải cho cùng xác suất (đã làm tròn 3 chữ số).

**Cách đọc.**
- **Con số:** chênh lớn nhất giữa hai cách **0,0000**. Thời gian cho 500 dòng: từng dòng **5,46** giây, cả bảng **0,0051** giây, chậm hơn khoảng **1076** lần.
- **Bức tranh:** kết quả giống nhau là điều bắt buộc kiểm (một cách có thể vô tình khác cách kia). Dự đoán từng dòng chậm hơn nhiều vì mỗi lần gọi có chi phí cố định (kiểm tra, dựng bảng nhỏ, tính lý do). Dự đoán hằng ngày cho cả cửa hàng nên dùng cách theo bảng; cách từng dòng dành cho tra cứu một khách tại quầy.
- **Quy tắc đọc:** viết **một** nguồn sự thật cho dự đoán; cách theo bảng và cách từng dòng phải được kiểm chéo bằng một bài kiểm tra tự động.

**Mang về chặng 4:** một dòng và nhiều dòng phải khớp nhau; dự đoán hằng loạt thì dùng theo bảng.

## Chặng 4: Ngưỡng quyết định theo chi phí

**Câu hỏi:** "nguy cơ không quay lại" khi xác suất dưới bao nhiêu? 0,5 có hợp lý không?

In [6]:
VOUCHER_COST, VALUE_IF_SAVED, SAVE_RATE = 2.0, 15.0, 0.30
def expected_profit(proba_returns, thresholds, y_true):
    out = {}
    for t in thresholds:
        target = proba_returns < t
        saved_value = np.where(y_true.to_numpy() == 0, VALUE_IF_SAVED * SAVE_RATE, 0.0)
        out[t] = float((saved_value[target] - VOUCHER_COST).sum())
    return pd.Series(out)
p_val = model.predict_proba(X_val)[:, 1]
p_test = model.predict_proba(X_test)[:, 1]
ths = np.round(np.arange(0.1, 0.95, 0.05), 2)
profit_val = expected_profit(p_val, ths, y_val)
best_t = float(profit_val.idxmax())
profit_test = expected_profit(p_test, [0.5, best_t], y_test)
print("voucher costs", VOUCHER_COST, "| value of a saved customer", VALUE_IF_SAVED, "| chance the voucher saves a would-be-lost customer", SAVE_RATE)
print("best threshold on the validation set:", best_t, "| profit there:", round(float(profit_val.max()), 1), "| at 0.5:", round(float(profit_val.loc[0.5]), 1))
print("on the test set -> threshold 0.5:", round(float(profit_test[0.5]), 1), "| chosen threshold", best_t, ":", round(float(profit_test[best_t]), 1))

voucher costs 2.0 | value of a saved customer 15.0 | chance the voucher saves a would-be-lost customer 0.3
best threshold on the validation set: 0.55 | profit there: 328.5 | at 0.5: 308.0
on the test set -> threshold 0.5: 357.5 | chosen threshold 0.55 : 360.0


**Giải thích từng dòng**

- `VOUCHER_COST, VALUE_IF_SAVED, SAVE_RATE`: ba con số nghiệp vụ: phiếu giảm giá tốn 2, giữ được một khách đáng 15, và phiếu chỉ cứu được 30% khách sắp rời bỏ.
- `target = proba_returns < t`: gửi phiếu cho khách có xác suất quay lại **dưới** ngưỡng `t`.
- `np.where(y_true == 0, VALUE * SAVE_RATE, 0)`: nếu khách thật sự sẽ không quay lại thì phiếu có 30% cơ hội cứu, giá trị kỳ vọng `15 x 0,3`; nếu khách vốn quay lại thì phiếu **không đem thêm gì**, vẫn mất 2.
- `profit_val.idxmax()`: chọn ngưỡng cho lợi nhuận kỳ vọng cao nhất **trên tập kiểm**, rồi báo cáo trên **tập thi**.

**Cách đọc.**
- **Con số:** ngưỡng tốt nhất trên tập kiểm là **0,55** (lợi nhuận kỳ vọng 328,5), còn ngưỡng 0,5 chỉ **308,0**. Trên tập thi: ngưỡng 0,5 được **357,5**, ngưỡng đã chọn **360,0**.
- **Bức tranh:** ngưỡng không phải con số "tự nhiên" 0,5; nó là **hệ quả của chi phí**: phiếu tốn 2 còn giá trị kỳ vọng khi cứu được khách là 15 x 0,3 = 4,5, nên ngưỡng tối ưu nằm gần 0,5 và chênh lệch lợi nhuận giữa các ngưỡng lân cận khá nhỏ (đường lợi nhuận khá phẳng quanh đỉnh). Đổi chi phí thì đổi ngưỡng.
- **Quy tắc đọc:** chọn ngưỡng trên tập kiểm, báo cáo trên tập thi (D1, D4); lưu ngưỡng và chi phí trong **file cấu hình** cạnh mô hình, để nghiệp vụ đổi được mà không phải huấn luyện lại.

**Mang về chặng 5:** ngưỡng là quyết định kinh doanh, lưu trong cấu hình.

## Chặng 5: Nhật ký và xử lý đầu vào sai

**Câu hỏi:** làm sao biết sau vài tháng hàm đã nhận gì và trả gì?

In [7]:
import os, json, tempfile, datetime
log_path = os.path.join(tempfile.mkdtemp(), "predictions.jsonl")
def predict_and_log(row, model_name="customer_return_v1", threshold=None):
    started = time.time()
    result = predict_one(row, threshold)
    entry = {"time": datetime.datetime.now().isoformat(timespec="seconds"), "model": model_name, "input": {k: (None if (isinstance(v, float) and np.isnan(v)) else v) for k, v in row.items()},
             "status": result["status"], "probability": result.get("probability_returns"), "warnings": result.get("warnings"), "errors": result.get("errors"), "ms": round((time.time() - started) * 1000, 2)}
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, "item") else str(o)) + "\n")
    return result
requests = [example, {**example, "city": "PQ"}, {**example, "tenure_months": -5}, risk, {k: v for k, v in example.items() if k != "plan"}]
for r in requests:
    predict_and_log(r)
log = pd.read_json(log_path, lines=True)
print(log[["status", "probability", "warnings", "errors", "ms"]].to_string())
print("share of refused requests:", round(float((log["status"] == "refused").mean()), 2))

    status  probability                   warnings                                errors     ms
0       ok        0.870                         []                                  None  13.19
1       ok        0.870  [city: unseen value 'PQ']                                  None  11.18
2  refused          NaN                       None  [tenure_months: -5 outside [0, 120]]   0.04
3       ok        0.248                         []                                  None  12.58
4  refused          NaN                       None                       [plan: missing]   0.03
share of refused requests: 0.4


**Giải thích từng dòng**

- `def predict_and_log(row, ...)`: bọc `predict_one`: mỗi lần gọi ghi một dòng nhật ký gồm thời gian, tên mô hình, đầu vào, kết quả, lỗi, thời gian xử lý.
- `"a" mode, json.dumps(entry)`: ghi nối thêm vào file định dạng `jsonl` (mỗi dòng một bản ghi JSON): dễ ghi liên tục, dễ đọc lại bằng `pandas`.
- `pd.read_json(log_path, lines=True)`: đọc nhật ký thành bảng để phân tích (ở H3 ta dùng nó để phát hiện dữ liệu đổi).
- `(log["status"] == "refused").mean()`: tỷ lệ yêu cầu bị từ chối: tăng đột ngột nghĩa là nơi nhập liệu có vấn đề.

**Cách đọc.**
- **Con số:** nhật ký có **5** dòng; **2** yêu cầu bị từ chối (tỷ lệ **40%**), **1** có cảnh báo. Thời gian xử lý mỗi yêu cầu cỡ **10,1** mili giây.
- **Bức tranh:** nhật ký là **mắt** của mô hình khi đã ra đời: sau vài tháng ta mới biết dữ liệu thật khác dữ liệu học ở đâu. Không ghi nhật ký thì H3 (giám sát) không có gì để nhìn.
- **Lưu ý riêng tư:** nhật ký chứa dữ liệu khách; với dữ liệu thật cần bỏ hoặc che thông tin nhận dạng, và giới hạn thời gian lưu.

**Mang về:** lược đồ, kiểm tra, trả lời có cấu trúc, ngưỡng theo chi phí, nhật ký: một hàm dự đoán đáng tin.

## Tổng kết

**Mang về theo từng chặng**

| Chặng | Mang về |
|---|---|
| 0 | Mô hình nuốt cả dữ liệu vô lý; phải có người gác cửa. |
| 1 | Lược đồ ghi rõ cột, kiểu, khoảng; phân biệt lỗi (từ chối) và cảnh báo. |
| 2 | Trả lời có xác suất, quyết định, lý do, cảnh báo. |
| 3 | Dự đoán một dòng và theo bảng phải khớp nhau; hằng loạt thì dùng theo bảng. |
| 4 | Ngưỡng là quyết định chi phí, lưu trong cấu hình. |
| 5 | Ghi nhật ký mọi yêu cầu để giám sát về sau. |

**Bảng đối chiếu: đời thường, thuật ngữ, tên trong code**

| Đời thường | Thuật ngữ | Tên trong code |
|---|---|---|
| Quầy lễ tân kiểm giấy tờ | kiểm tra đầu vào | `validate_row` |
| Khoảng hợp lý theo nghiệp vụ | lược đồ | `SCHEMA` |
| Từ chối hẳn hoặc trả lời kèm lưu ý | lỗi và cảnh báo | `errors`, `warnings` |
| Vì sao mô hình đoán thế | lý do chính | `coef * giá trị chuẩn hóa` |
| Gửi phiếu khi nào có lời | ngưỡng theo chi phí | `expected_profit` |
| Sổ ghi mọi cuộc gọi | nhật ký dự đoán | `predictions.jsonl` |

**Thử thách tự làm**

1. Ở chặng 1, thêm quy tắc: nếu `visits_last_30d` > 40 thì **cảnh báo** (không từ chối). Kiểm tra bằng một dòng mẫu.
2. Ở chặng 4, đổi `VOUCHER_COST` thành 6. Ngưỡng tốt nhất dịch đi đâu? Vì sao?
3. Ở chặng 2, thêm khóa `confidence` = 'low' khi xác suất nằm trong khoảng 0,4 đến 0,6.
4. Ở chặng 5, ghi thêm `schema_version` vào nhật ký. Lợi ích khi lược đồ đổi sau này là gì?

**Checklist năng lực** (tự tick khi làm được)

- [ ] Viết lược đồ đầu vào từ hiểu biết nghiệp vụ.
- [ ] Phân biệt lỗi và cảnh báo.
- [ ] Trả lời có xác suất, quyết định, lý do, cảnh báo.
- [ ] Chọn ngưỡng theo chi phí trên tập kiểm, báo cáo trên tập thi.
- [ ] Ghi nhật ký dự đoán có đủ thông tin để theo dõi.

## Phần thưởng (Tip pro): Sang H3: giám sát khi dữ liệu đổi


Hàm đã chạy, nhật ký đã ghi. Nhưng khách hàng, giá cả, mùa vụ thay đổi theo tháng; mô hình học từ quá khứ sẽ **xuống cấp** dần. H3 dạy phát hiện dữ liệu trôi và quyết định khi nào cần học lại.